In [1]:
pip install dash pandas plotly openpyxl

  Using cached zipp-3.21.0-py3-none-any.whl.metadata (3.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 15.2 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 12.1 MB/s eta 0:00:00
Using cached zipp-3.21.0-py3-none-any.whl (9.6 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install dash-bootstrap-components


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.3/229.3 kB 4.5 MB/s eta 0:00:006.1 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [22]:
import dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State
import pandas as pd
import plotly.express as px
import base64
import io
import datetime

In [23]:
import base64
import datetime
import io
import re

import dash
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.express as px
import pycountry
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State
from datetime import datetime as dt
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

# =============================================================================
# Fonctions de contrôle qualité
# =============================================================================

def check_freshness(df, date_column, threshold_years=2):
    """Analyse la fraîcheur des données en comparant une date de référence."""
    today = dt.today()
    df[date_column] = pd.to_datetime(df[date_column], errors='coerce')
    df['Obsolete'] = (today - df[date_column]).dt.days > (threshold_years * 365)
    obsolete_count = df['Obsolete'].sum()
    summary = df[['Obsolete', date_column]].value_counts().reset_index(name='Count')
    return summary, obsolete_count

def check_missing_data(df, required_columns):
    """Analyse des données manquantes pour les colonnes sélectionnées."""
    missing_report = {}
    for col in required_columns:
        missing_count = df[col].isnull().sum()
        missing_percentage = (missing_count / len(df)) * 100
        missing_report[col] = {"missing_count": missing_count, "missing_percentage": missing_percentage}
    return pd.DataFrame(missing_report).T

def validate_postal_code(df, postal_code_column, valid_length=5):
    """Valide la longueur des codes postaux."""
    df['Invalid_Code'] = df[postal_code_column].apply(
        lambda x: len(str(x)) != valid_length if pd.notnull(x) else True
    )
    invalid_count = df['Invalid_Code'].sum()
    return df[df['Invalid_Code']], invalid_count

def validate_phone_number(df, phone_column):
    """Valide les numéros de téléphone au format E.164."""
    phone_pattern = r'^\+?[1-9]\d{1,14}$'
    df['Invalid_Phone'] = df[phone_column].apply(
        lambda x: not re.match(phone_pattern, str(x)) if pd.notnull(x) else True
    )
    invalid_count = df['Invalid_Phone'].sum()
    return df[df['Invalid_Phone']], invalid_count

def validate_nom_prenom(df, nom_column):
    """Valide que le nom ne contient que des lettres et des espaces."""
    nom_pattern = r'^[A-Za-z\s]+$'
    df['Invalid_Nom'] = df[nom_column].apply(
        lambda x: not re.match(nom_pattern, str(x)) if pd.notnull(x) else True
    )
    invalid_count = df['Invalid_Nom'].sum()
    return df[df['Invalid_Nom']], invalid_count

def validate_continent(df, continent_column):
    """Valide la présence d'un continent dans la liste des continents attendus."""
    valid_continents = ["Africa", "Asia", "Europe", "North America", 
                        "South America", "Oceania", "Antarctica", "Afrique",
                        "Asie", "Europe", "Amérique du nord", "Amérique du sud", "Australie"]
    df['Invalid_Continent'] = df[continent_column].apply(
        lambda x: x.strip().capitalize() not in valid_continents if pd.notnull(x) else True
    )
    invalid_count = df['Invalid_Continent'].sum()
    return df[df['Invalid_Continent']], invalid_count

def validate_pays(df, pays_column):
    """Valide que le pays saisi figure dans la liste des pays ISO."""
    valid_pays = [country.name for country in pycountry.countries]
    df['Invalid_Pays'] = df[pays_column].apply(
        lambda x: x.strip().title() not in valid_pays if pd.notnull(x) else True
    )
    invalid_count = df['Invalid_Pays'].sum()
    return df[df['Invalid_Pays']], invalid_count

def validate_ville(df, ville_column):
    """Valide l'existence d'une ville via le géocodage."""
    geolocator = Nominatim(user_agent="city_validator")
    def is_valid_ville(ville):
        try:
            location = geolocator.geocode(ville, timeout=10)
            return location is not None
        except GeocoderTimedOut:
            return False

    df['Invalid_Ville'] = df[ville_column].apply(
        lambda x: not is_valid_ville(x) if pd.notnull(x) else True
    )
    invalid_count = df['Invalid_Ville'].sum()
    return df[df['Invalid_Ville']], invalid_count

def validate_email(df, email_column):
    """Valide les adresses email par un pattern basique."""
    email_pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    df['Invalid_Email'] = df[email_column].apply(
        lambda x: not re.match(email_pattern, str(x)) if pd.notnull(x) else True
    )
    invalid_count = df['Invalid_Email'].sum()
    return df[df['Invalid_Email']], invalid_count

def validate_date_naissance(df, date_column, date_format='%Y-%m-%d'):
    """Valide les dates de naissance en s'assurant qu'elles sont antérieures à aujourd'hui."""
    def is_valid_date(date):
        try:
            parsed_date = dt.strptime(date, date_format)
            return parsed_date <= dt.now()
        except (ValueError, TypeError):
            return False

    df['Invalid_Date_Naissance'] = df[date_column].apply(
        lambda x: not is_valid_date(x) if pd.notnull(x) else True
    )
    invalid_count = df['Invalid_Date_Naissance'].sum()
    return df[df['Invalid_Date_Naissance']], invalid_count

def validate_date_entree_relation(df, date_column, date_format='%Y-%m-%d'):
    """Valide les dates d'entrée en relation en vérifiant leur format."""
    def is_valid_date(date):
        try:
            dt.strptime(date, date_format)
            return True
        except (ValueError, TypeError):
            return False

    df['Invalid_Date_Entree_Relation'] = df[date_column].apply(
        lambda x: not is_valid_date(x) if pd.notnull(x) else True
    )
    invalid_count = df['Invalid_Date_Entree_Relation'].sum()
    return df[df['Invalid_Date_Entree_Relation']], invalid_count

def validate_adresse_postale(df, adresse_column):
    """Valide le format des adresses postales via une expression régulière."""
    # Exemple de pattern simplifié
    adresse_pattern = r'\d+,\s+[A-Za-z]+\s+[A-Za-z]+'
    df['Invalid_Adresse'] = df[adresse_column].apply(
        lambda x: not re.match(adresse_pattern, str(x)) if pd.notnull(x) else True
    )
    invalid_count = df['Invalid_Adresse'].sum()
    return df[df['Invalid_Adresse']], invalid_count

# =============================================================================
# Initialisation de l'application Dash
# =============================================================================

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP],
                suppress_callback_exceptions=True)
app.title = "Contrôle Qualité des Données"

# Stockage global des données
app_data = {'df': None}

# =============================================================================
# Layout de l'application
# =============================================================================

# Sidebar avec chargement de fichier, choix d'analyse et validations
sidebar = dbc.Col([
    html.H2("Contrôle Qualité des Données", className="text-center"),
    dcc.Upload(
        id='upload-data',
        children=dbc.Button("Choisir un fichier (CSV ou Excel)", color="primary", className="mt-2"),
        style={'textAlign': 'center'}
    ),
    html.Div(id='file-info', className="mt-2"),
    html.Hr(),
    html.H4("Graphique"),
    dcc.Dropdown(
        id='graph-type',
        options=[
            {'label': 'Histogramme', 'value': 'Histogramme'},
            {'label': 'Boxplot', 'value': 'Boxplot'},
            {'label': 'Camembert', 'value': 'Camembert'}
        ],
        placeholder="Choisir un type de graphique"
    ),
    dcc.Dropdown(id='graph-col', multi=True, placeholder="Sélectionner les colonnes"),
    dbc.Button("Tracer", id='plot-graph', color="primary", className="mt-2"),
    html.Hr(),
    html.H4("Validation / Contrôle"),
    dcc.Dropdown(
        id='validation-type',
        options=[
            {'label': 'Données manquantes', 'value': 'missing'},
            {'label': 'Fraîcheur', 'value': 'freshness'},
            {'label': 'Code Postal', 'value': 'postal'},
            {'label': 'Téléphone', 'value': 'phone'},
            {'label': 'Nom/Prénom', 'value': 'nom'},
            {'label': 'Continent', 'value': 'continent'},
            {'label': 'Pays', 'value': 'pays'},
            {'label': 'Ville', 'value': 'ville'},
            {'label': 'Email', 'value': 'email'},
            {'label': 'Date de Naissance', 'value': 'date_naissance'},
            {'label': "Date d'entrée en relation", 'value': 'date_entree'},
            {'label': 'Adresse Postale', 'value': 'adresse'},
        ],
        placeholder="Sélectionner le type de validation"
    ),
    dcc.Dropdown(id='validation-col', placeholder="Sélectionner la colonne concernée"),
    dbc.Button("Valider", id='run-validation', color="secondary", className="mt-2")
], width=3, className="bg-light p-3")

# Contenu principal avec aperçu des données, résultats graphiques et résultats de validation
content = dbc.Col([
    dbc.Row([
        dbc.Col(dbc.Card([
            dbc.CardHeader("Graphique Généré"),
            dbc.CardBody(dcc.Graph(id='graph-output'))
        ]), width=12)
    ], className="mb-3"),
    html.H3("Données Chargées"),
    dash_table.DataTable(id='data-preview', page_size=5, style_table={'overflowX': 'auto'}),
    html.Hr(),
    html.H3("Résultat de la Validation"),
    html.Div(id='validation-result')
], width=9)

app.layout = dbc.Container([
    dbc.Row([sidebar, content])
], fluid=True)

# =============================================================================
# Fonctions utilitaires
# =============================================================================

def parse_data(contents, filename):
    """Décodage et lecture du fichier uploadé (CSV ou Excel)."""
    content_type, content_string = contents.split(',')
    decoded = base64.b64decode(content_string)
    ext = filename.split('.')[-1].lower()
    if ext == 'csv':
        return pd.read_csv(io.StringIO(decoded.decode('utf-8')))
    elif ext in ['xls', 'xlsx']:
        return pd.read_excel(io.BytesIO(decoded))
    else:
        return None

# Mise à jour des données chargées et des options de colonnes
@app.callback(
    [Output('file-info', 'children'),
     Output('data-preview', 'data'),
     Output('data-preview', 'columns'),
     Output('graph-col', 'options'),
     Output('validation-col', 'options')],
    [Input('upload-data', 'contents')],
    [State('upload-data', 'filename')]
)
def update_data(contents, filename):
    if contents is None:
        return '', [], [], [], []
    df = parse_data(contents, filename)
    if df is None:
        return 'Format de fichier non supporté', [], [], [], []
    app_data['df'] = df
    columns = [{'name': col, 'id': col} for col in df.columns]
    options = [{'label': col, 'value': col} for col in df.columns]
    return f'Fichier chargé : {filename}', df.to_dict('records'), columns, options, options

# Génération du graphique selon le type choisi
@app.callback(
    Output('graph-output', 'figure'),
    [Input('plot-graph', 'n_clicks')],
    [State('graph-type', 'value'),
     State('graph-col', 'value')]
)
def generate_graph(n_clicks, graph_type, selected_cols):
    if not n_clicks or app_data['df'] is None or not selected_cols:
        return px.scatter()
    df = app_data['df']
    if graph_type == 'Histogramme':
        fig = px.histogram(df, x=selected_cols[0])
    elif graph_type == 'Boxplot':
        fig = px.box(df, y=selected_cols[0])
    elif graph_type == 'Camembert':
        fig = px.pie(df, names=selected_cols[0])
    else:
        fig = px.scatter()
    return fig

# Exécution de la validation sélectionnée
@app.callback(
    Output('validation-result', 'children'),
    [Input('run-validation', 'n_clicks')],
    [State('validation-type', 'value'),
     State('validation-col', 'value')]
)
def run_validation(n_clicks, val_type, col):
    if not n_clicks or app_data['df'] is None or not val_type or not col:
        return ""
    df = app_data['df']
    result_table = None
    summary = ""
    # Validation selon le type sélectionné
    if val_type == 'missing':
        res = check_missing_data(df, [col])
        summary = f"Nombre de valeurs manquantes dans '{col}' : {res['missing_count'].values[0]} "\
                  f"({res['missing_percentage'].values[0]:.2f}%)"
    elif val_type == 'freshness':
        res, obsolete_count = check_freshness(df, col)
        summary = f"Nombre d'enregistrements obsolètes dans '{col}' : {obsolete_count}"
        result_table = res.to_dict('records')
    elif val_type == 'postal':
        res, count = validate_postal_code(df, col)
        summary = f"Nombre de codes postaux invalides dans '{col}' : {count}"
        result_table = res.to_dict('records')
    elif val_type == 'phone':
        res, count = validate_phone_number(df, col)
        summary = f"Nombre de numéros de téléphone invalides dans '{col}' : {count}"
        result_table = res.to_dict('records')
    elif val_type == 'nom':
        res, count = validate_nom_prenom(df, col)
        summary = f"Nombre de noms invalides dans '{col}' : {count}"
        result_table = res.to_dict('records')
    elif val_type == 'continent':
        res, count = validate_continent(df, col)
        summary = f"Nombre de continents invalides dans '{col}' : {count}"
        result_table = res.to_dict('records')
    elif val_type == 'pays':
        res, count = validate_pays(df, col)
        summary = f"Nombre de pays invalides dans '{col}' : {count}"
        result_table = res.to_dict('records')
    elif val_type == 'ville':
        res, count = validate_ville(df, col)
        summary = f"Nombre de villes invalides dans '{col}' : {count}"
        result_table = res.to_dict('records')
    elif val_type == 'email':
        res, count = validate_email(df, col)
        summary = f"Nombre d'emails invalides dans '{col}' : {count}"
        result_table = res.to_dict('records')
    elif val_type == 'date_naissance':
        res, count = validate_date_naissance(df, col)
        summary = f"Nombre de dates de naissance invalides dans '{col}' : {count}"
        result_table = res.to_dict('records')
    elif val_type == 'date_entree':
        res, count = validate_date_entree_relation(df, col)
        summary = f"Nombre de dates d'entrée en relation invalides dans '{col}' : {count}"
        result_table = res.to_dict('records')
    elif val_type == 'adresse':
        res, count = validate_adresse_postale(df, col)
        summary = f"Nombre d'adresses invalides dans '{col}' : {count}"
        result_table = res.to_dict('records')
    else:
        summary = "Type de validation non reconnu."

    # Construction de l'affichage
    result_display = [html.H5(summary)]
    if result_table is not None and len(result_table) > 0:
        result_display.append(
            dash_table.DataTable(
                data=result_table,
                columns=[{"name": i, "id": i} for i in result_table[0].keys()],
                page_size=5,
                style_table={'overflowX': 'auto'}
            )
        )
    return result_display

if __name__ == '__main__':
    app.run_server(debug=True)


/home/cytech/Desktop/DATA QUALITY/env/lib/python3.12/site-packages/dash/dash.py:2282: DeprecationWarning:

Dash.run_server is deprecated and will be removed in Dash 3.0

/home/cytech/Desktop/DATA QUALITY/env/lib/python3.12/site-packages/dash/dash.py:1814: DeprecationWarning:

'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead



In [6]:
python app.py

SyntaxError: invalid syntax (945115591.py, line 1)

In [ ]:
#missing data

def missing_data(colonne, df):
    """
    Retourne l'ensemble des index correspondant aux données manquantes de la colonne spécifiée.

    Args:
        colonne (str): Nom de la colonne à analyser.
        df (pandas.DataFrame): DataFrame contenant les données.

    Returns:
        list: Liste des index pour lesquels les données sont manquantes dans la colonne spécifiée.
    """
    return colonne, df[df[colonne].isnull()].index.tolist()




In [ ]:
from datetime import datetime
import re
class ClientError(Exception):
    """Custom exception class for client data validation errors"""
    
    # Error codes
    MISSING_VALUE = 1
    INVALID_TYPE = 2
    INVALID_FORMAT = 3
    
    def __init__(self, error_code, field_name, message=None):
        """
        Initialize client error
        
        Args:
            error_code (int): Error type code
            field_name (str): Name of the field that caused the error
            message (str, optional): Custom error message
        """
        self.error_code = error_code
        self.field_name = field_name
        
        # Default messages based on error type
        if message is None:
            if error_code == self.MISSING_VALUE:
                message = f"Le champ '{field_name}' est obligatoire"
            elif error_code == self.INVALID_TYPE:
                message = f"Type invalide pour le champ '{field_name}'"
            elif error_code == self.INVALID_FORMAT:
                message = f"Format invalide pour le champ '{field_name}'"
            else:
                message = f"Erreur de validation pour le champ '{field_name}'"
                
        self.message = message
        super().__init__(self.message)
    
    def __str__(self):
        return f"Erreur {self.error_code}: {self.message}"
    
class Client:
    def __init__(self, id_client, nom, prenom, date_naissance, date_entree_relation, 
                 adresse_postale, ville, code_postale, email):
        # Validate and set id_client
        if not isinstance(id_client, str) or not id_client:
            raise ValueError("ID client must be a non-empty string")
        self.id_client = id_client

        # Validate and set name fields
        if not isinstance(nom, str) or not nom.strip():
            raise ValueError("Nom must be a non-empty string")
        self.nom = nom.strip()

        if not isinstance(prenom, str) or not prenom.strip():
            raise ValueError("Prenom must be a non-empty string")
        self.prenom = prenom.strip()

        # Validate and set dates
        try:
            self.date_naissance = datetime.strptime(date_naissance, '%Y-%m-%d')
            self.date_entree_relation = datetime.strptime(date_entree_relation, '%Y-%m-%d')
            if self.date_entree_relation < self.date_naissance:
                raise ValueError("Date d'entrée en relation cannot be before date de naissance")
        except ValueError as e:
            raise ValueError("Invalid date format. Use YYYY-MM-DD") from e

        # Validate and set address fields
        if not isinstance(adresse_postale, str) or not adresse_postale.strip():
            raise ValueError("Adresse postale must be a non-empty string")
        self.adresse_postale = adresse_postale.strip()

        if not isinstance(ville, str) or not ville.strip():
            raise ValueError("Ville must be a non-empty string")
        self.ville = ville.strip()

        # Validate French postal code format (5 digits)
        if not re.match(r'^\d{5}$', str(code_postale)):
            raise ValueError("Code postal must be exactly 5 digits")
        self.code_postale = code_postale

        # Validate email format
        email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        if not re.match(email_pattern, email):
            raise ValueError("Invalid email format")
        self.email = email

    def __str__(self):
        return f"{self.prenom} {self.nom} (ID: {self.id_client})"

In [19]:
# Import required modules from GX library.
import great_expectations as gx
import great_expectations.expectations as gxe
import pandas as pd

# Create Data Context.
context = gx.get_context()

# Import sample data into Pandas DataFrame.
df = pd.read_csv("clients_data_quality_updated.csv")

# Connect to data.
# Create Data Source, Data Asset, Batch Definition, and Batch.
data_source = context.data_sources.add_pandas("pandas")
data_asset = data_source.add_dataframe_asset(name="pd dataframe asset")

batch_definition = data_asset.add_batch_definition_whole_dataframe("batch definition")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})

# Create Expectation.
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="ID Client", min_value=1, max_value=6
)

# Validate Batch using Expectation.
validation_result = batch.validate(expectation)

gxe.ExpectColumnValuesToBeNull(column="errors")

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

ExpectColumnValuesToBeNull(id=None, meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, windows=None, batch_id=None, column='errors', mostly=1, row_condition=None, condition_parser=None)

In [29]:
import streamlit as st
import pandas as pd
import great_expectations as ge

# Configurer la page Streamlit
st.set_page_config(page_title="Moteur de Qualité des Données", layout="wide")

st.title("📊 Moteur de Qualité des Données en Python")
st.write("Téléchargez un fichier CSV et effectuez une analyse de qualité des données.")

# 📂 Upload du fichier CSV
uploaded_file = st.file_uploader("Téléchargez un fichier CSV", type=["csv"])

if uploaded_file:
    # Lecture des données
    df = pd.read_csv(uploaded_file)
    st.write("✅ **Aperçu des données :**")
    st.dataframe(df.head())

    # Conversion en DataFrame Great Expectations
    df_ge = ge.from_pandas(df)

    # 📌 Définition des contrôles de qualité
    st.subheader("🔍 Analyse de la Qualité des Données")

    checks = {
        "Valeurs manquantes": df.isnull().sum().sum(),
        "Doublons": df.duplicated().sum(),
        "Colonnes avec valeurs nulles": df.isnull().sum().to_dict(),
    }

    # 🔹 Vérification des valeurs manquantes
    missing_values = df.isnull().sum()
    for col, count in missing_values.items():
        if count > 0:
            st.warning(f"⚠️ {count} valeurs manquantes dans la colonne **'{col}'**.")

    # 🔹 Vérification des doublons
    if df.duplicated().sum() > 0:
        st.warning(f"⚠️ {df.duplicated().sum()} doublons détectés.")

    # 🔹 Contrôle sur certaines colonnes
    if "Age" in df.columns:
        df_ge.expect_column_values_to_be_between("Age", 18, 100)
    if "Email" in df.columns:
        df_ge.expect_column_values_to_match_regex("Email", r"^[\w\.-]+@[\w\.-]+\.\w+$")

    # 📌 Validation et résultats
    results = df_ge.validate()
    success = results["success"]

    if success:
        st.success("✅ Aucune anomalie détectée, les données sont conformes !")
    else:
        st.error("❌ Problèmes détectés dans les données !")

    # 📜 Générer un rapport JSON
    import json
    report_path = "rapport_qualite.json"
    with open(report_path, "w") as f:
        json.dump(results, f, indent=4)

    st.download_button(
        label="📥 Télécharger le rapport de qualité",
        data=json.dumps(results, indent=4),
        file_name="rapport_qualite.json",
        mime="application/json",
    )


2025-02-23 10:09:20.640 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-23 10:09:20.642 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-23 10:09:20.738 
  command:

    streamlit run /home/cytech/Desktop/DATA QUALITY/env/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-02-23 10:09:20.739 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-23 10:09:20.740 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-23 10:09:20.741 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-23 10:09:20.741 Thread 'MainThread': missing ScriptRunContext! This warning 

In [ ]:

streamlit run /home/cytech/Desktop/DATA QUALITY/env/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
